In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
OUTPUT_DIR = ROOT / "working/analysis/stats-rework/out/trajectories"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
### this code is written to plot survival data

In [ ]:
import pandas as pd
import numpy as np
from matplotlib.ticker import LogFormatter
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns 
import json

In [ ]:
# Assign Data to path

PROJECT_PATH = ROOT / 'working/figures/Supplemental Figure 4 - Pb/A - Survival'
file= 'SurvivalData.xlsx'
PROJECT_PATH


In [ ]:
# Read the Excel file into a DataFrame
all_sheets_data = pd.read_excel(PROJECT_PATH / file, sheet_name=None, engine='openpyxl')
print("Sheet names:", all_sheets_data.keys())


In [ ]:
# Specify the selected sheet name
selected_evolution = 'ATEC'  # Replace 'PC' with your desired sheet name

# Extract the data for the selected sheet
if selected_evolution in all_sheets_data:
    select_survival_data = all_sheets_data[selected_evolution]
    print("Data from the selected sheet:")
    print(select_survival_data.head())  # Display the first few rows
else:
    print(f"Sheet '{selected_evolution}' does not exist in the file.")


In [ ]:
filtered_data = select_survival_data[~select_survival_data['Culture'].isin(['A', 'B', 'C', 'D'])]


In [ ]:
# Filter the DataFrame to include only rows where Strain is 'MG1655'
data = filtered_data[filtered_data['Strain'] == 'ATEC']




In [ ]:
# Add a new column 'treated' to indicate 'before' or 'after'
data['treated'] = data['Day'].apply(
    lambda x: 'before' if x.is_integer() else 'after'
)
data.loc[data['treated'] == 'after', 'Day'] -= 0.5

# Display the updated DataFrame
print(data.tail())


In [ ]:
# Group by the specified columns
grouped = data.groupby(['Evolution', 'Day', 'Strain', 'Culture'])

# Calculate percent survival
def calculate_percent_survival(group):
    # Separate 'before' and 'after' data
    before = group[group['treated'] == 'before']
    after = group[group['treated'] == 'after']
    
    if not before.empty and not after.empty:
        # Take the first value of CFU from 'before' for calculation
        before_cfu = before['CFU'].iloc[0]
        # Add the percent survival column to the 'after' group
        after['PercentSurvival'] = (after['CFU'] / before_cfu) * 100
    
    return pd.concat([before, after])

# Apply the function to each group
result = grouped.apply(calculate_percent_survival).reset_index(drop=True)

# Display the updated DataFrame
print(result.head())


In [ ]:
# Filter the result to include only rows where 'treated' is 'after'
result = result[result['treated'] == 'after']
result = result.sort_values(by=["Culture", "Day"])

# Display the filtered DataFrame
print(result.head())
result

In [ ]:
# Subset the data for Evolution = "PL"
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"

# Calculate the mean PercentSurvival for each Day and Drug
mean_data = result.groupby(['Evolution', 'Day'])['PercentSurvival'].mean().reset_index()

# Create a FacetGrid to plot 'PercentSurvival' vs. 'Day', faceted by 'Drug'
g = sns.FacetGrid(result, col="Evolution", hue="Culture", col_wrap=4, sharey=False)
g.map(plt.plot, "Day", "PercentSurvival", color="gray", linewidth = 5, alpha = .5)  # Set line color to gray

# Add mean PercentSurvival in purple on top
for ax, drug in zip(g.axes.flat, g.col_names):
    mean_subset = mean_data[mean_data['Evolution'] == drug]
    ax.plot(mean_subset['Day'], mean_subset['PercentSurvival'], color="forestgreen", linewidth=10, label='Mean')
    
    # Make the axis lines bolder
    for spine in ax.spines.values():
        spine.set_linewidth(2)  # Set axis line thickness
    
    # Make tick marks larger and bolder
    ax.tick_params(axis='both', which='major', length=12, width=3)  # Major ticks
    ax.tick_params(axis='both', which='minor', length=8, width=2)  # Minor ticks

    # Set the y-axis to a log10 scale
    ax.set_yscale('log')
    ax.set_ylim(0.0005, 3000)  # Set y-axis limits
    ax.set_yticks([0.001, 0.01, 0.1, 1, 10, 100, 1000])  # Specify y-tick values
    ax.set_yticklabels(["0.001","0.01", "0.1", "1", "10", "100", "1000"])  # Specify y-tick labels
    ax.tick_params(axis='y', labelsize=35)  # Set y-tick font size

    # Set x-axis ticks and labels to show only even days
    even_days = result['Day'].unique()[result['Day'].unique() % 2 == 0]  # Filter even days
    ax.set_xticks(even_days)  # Set x-ticks to even Day values
    ax.set_xticklabels(even_days.astype(int), fontsize=35)  # Set x-tick labels for even days
    ax.set_xlim(result['Day'].min(), result['Day'].max())


for spine in ax.spines.values():
    spine.set_linewidth(4)  # Increase from 2 to 5 or another desired value
# Add axis labels
g.set_axis_labels("Day", "Survival (%)", fontsize=40, ) 


# Set titles dynamically to the value of "Drug"
g.set_titles("Evolution to {col_name}")  # Dynamically set the title to the value of the 'Drug' column

# Adjust font sizes for subplot titles
for ax in g.axes.flat:
    ax.title.set_fontsize(40)

# Set the overall figure size (width, height in inches)
g.fig.set_size_inches(40, 10)  # Predetermined figure size for consistency

# Adjust the spacing between subplots
g.fig.subplots_adjust(top=0.9, wspace=0, hspace=0.4)

g.fig.savefig(OUTPUT_DIR / f'PbCef_ps', dpi=600, bbox_inches='tight')
# Show the plot
plt.show()
